# Smoking verifier (yolo11n-cls) - train on Colab

ตัวยืนยันชั้นที่สองของ module smoking: แจ้งเตือนได้เมื่อ detector เดิม (`models/smoking/best.pt`) เจอ **และ** classifier นี้บอกว่า "smoking"

**กฎของข้อมูล (ห้ามเปลี่ยน)**
- อัปโหลด **เฉพาะโฟลเดอร์ `Training`** ของ Mendeley Smoker Detection ขึ้น Google Drive
- `Validation` (เลือก threshold) และ `Testing` (วัดผลก่อน/หลัง) ใช้บน Mac เท่านั้น ด้วย `tools/eval_smoking_cls.py` ห้ามอัปโหลด ห้ามใช้ที่นี่
- notebook นี้แบ่ง Training เป็น train 85% / val 15% (seed 0) เพื่อเลือก epoch เท่านั้น ตัวเลข val ที่นี่ไม่ใช่ผลวัดของโปรเจกต์

**ก่อนรัน:** Runtime → Change runtime type → **T4 GPU** แล้วแก้ `DRIVE_DIR` ในเซลล์ที่ 2 ให้ตรงกับที่อัปโหลด


In [ ]:
# 1. same Ultralytics version as the project Mac (8.4.136), then check the GPU
!pip install -q ultralytics==8.4.136
!nvidia-smi --query-gpu=name,memory.total --format=csv
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
assert ultralytics.__version__ == "8.4.136", "use the same version as the Mac"


In [ ]:
# 2. Google Drive: DRIVE_DIR must contain the Training folder ONLY
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/nursery_smoking')   # <- แก้ให้ตรงกับที่อัปโหลด
TRAIN_DIR = DRIVE_DIR / 'Training'
OUT_DIR = DRIVE_DIR / 'smoking_cls_v1'
assert TRAIN_DIR.is_dir(), f'{TRAIN_DIR} not found'
for other in ('Validation', 'Testing'):
    if (DRIVE_DIR / other).exists():
        print(f'WARNING: {DRIVE_DIR / other} exists - it is NOT used here and should not be uploaded')
print('Training images:', len(list(TRAIN_DIR.glob('*.jpg'))))


In [ ]:
# 3. class folders + 85/15 split of Training (seed 0, per class)
import random, shutil
SEED = 0
DS = Path('/content/ds')
if DS.exists():
    shutil.rmtree(DS)

def label(name):
    n = name.lower()
    if n.startswith('notsmoking_'): return 'notsmoking'
    if n.startswith('smoking_'): return 'smoking'
    raise ValueError(name)

by_class = {'smoking': [], 'notsmoking': []}
for p in sorted(TRAIN_DIR.glob('*.jpg')):
    by_class[label(p.name)].append(p)

counts = {}
rng = random.Random(SEED)
for cls, files in by_class.items():
    files = files[:]
    rng.shuffle(files)
    n_val = round(len(files) * 0.15)
    for split, part in (('val', files[:n_val]), ('train', files[n_val:])):
        dest = DS / split / cls
        dest.mkdir(parents=True, exist_ok=True)
        for f in part:
            shutil.copy2(f, dest / f.name)
        counts[f'{split}/{cls}'] = len(part)
print(counts)


In [ ]:
# 4. train yolo11n-cls (deterministic, early stop on the internal val split)
from ultralytics import YOLO
model = YOLO('yolo11n-cls.pt')   # ImageNet-pretrained classifier from Ultralytics
ARGS = dict(data=str(DS), epochs=30, imgsz=224, batch=32, seed=SEED, deterministic=True,
            patience=10, project='/content/runs', name='smoking_cls_v1', exist_ok=True)
model.train(**ARGS)


In [ ]:
# 5. save the weights + training record to Drive (download them to models/smoking_cls/ on the Mac)
import datetime, hashlib, json
run = Path('/content/runs/smoking_cls_v1')
OUT_DIR.mkdir(parents=True, exist_ok=True)
best = OUT_DIR / 'smoking_cls_v1.pt'
shutil.copy2(run / 'weights' / 'best.pt', best)
for f in ('results.csv', 'args.yaml'):
    if (run / f).exists():
        shutil.copy2(run / f, OUT_DIR / f)

sha = hashlib.sha256(best.read_bytes()).hexdigest()
info = {
    'file': best.name, 'sha256': sha,
    'date': datetime.datetime.now().isoformat(timespec='minutes'),
    'base': 'yolo11n-cls.pt', 'classes': YOLO(str(best)).names,
    'data': 'Mendeley Smoker Detection - Training only, split 85/15 with seed 0',
    'split_counts': counts, 'train_args': {k: v for k, v in ARGS.items() if k != 'data'},
    'ultralytics': ultralytics.__version__, 'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
}
(OUT_DIR / 'train_info.json').write_text(json.dumps(info, indent=2, ensure_ascii=False))
print(json.dumps(info, indent=2, ensure_ascii=False))
print('\nDownload the folder', OUT_DIR, 'and put smoking_cls_v1.pt + train_info.json in models/smoking_cls/')


## ขั้นต่อไปบน Mac (ไม่ใช่ที่นี่)

```bash
python tools/eval_smoking_cls.py select --dataset <Smoker Detection> --classifier models/smoking_cls/smoking_cls_v1.pt   # Validation
python tools/eval_smoking_cls.py report --dataset <Smoker Detection> --classifier models/smoking_cls/smoking_cls_v1.pt   # Testing
```
ใช้จริงในระบบได้ก็ต่อเมื่อ `report` บอก ADOPT **และ** ผลบนคลิป QA ไม่แย่ลง
